# Обработка трёх наборов временных рядов

Сначала выполните `Download_public_ds.ipynb`. Этот ноутбук читает уже скачанные данные
и синтетику, задаёт настройки обработки, запускает функции и показывает результат.

Вспомогательные функции находятся в `src/prepare_datasets.py`, а финансовый обработчик —
в `src/prepare_moex.py`. Оба файла должны лежать рядом в папке `src`.

**Одна строка `series.csv` — один момент времени, числовые столбцы — вершины графа.**
У каждой вершины один признак: состояние синтетики, скорость METR-LA или log(RV) акции.
История из 60 наблюдений собирается функцией `get_window()` при загрузке.

Необходимы `numpy`, `pandas`, `tables` (для HDF5 METR-LA).
При необходимости выполните в выбранном ядре `%pip install numpy pandas tables`.


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# Change PROJECT_ROOT if the kernel starts outside the project.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = PROJECT_ROOT.resolve()

# Import reusable functions from the project's src directory.
SRC_DIR = PROJECT_ROOT / "src"
for filename in ("prepare_datasets.py", "prepare_moex.py"):
    if not (SRC_DIR / filename).is_file():
        raise FileNotFoundError(f"Place {filename} in {SRC_DIR}")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from prepare_datasets import (
    SPLITS, export_dataset, export_moex, get_window, load_metr_la, load_synthetic,
)

results = {}

PROCESS_SYNTHETIC = True
PROCESS_METR_LA = True
PROCESS_MOEX = True

SYNTHETIC_CSV = PROJECT_ROOT / "data/processed/synthetic_static.csv"
SYNTHETIC_TIME_COLUMN = "step"
METR_LA_H5 = PROJECT_ROOT / "data/raw/metr_la/metr-la.h5"
MOEX_RAW_ROOT = PROJECT_ROOT / "data/raw/moex_iss_2020_2025"
OUTPUT_ROOT = PROJECT_ROOT / "data/processed"

HISTORY = 60
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.10
MOEX_TRAIN_END = "2023-12-31"
MOEX_VAL_END = "2024-12-31"
MOEX_TEST_END = "2025-12-31"
MOEX_N_STOCKS = 10
MOEX_MIN_COVERAGE = 0.95
MOEX_MAX_GAP_DAYS = 7
MOEX_LOG_EPSILON = 1e-12
MOEX_RUN_ID = None

print("Корень проекта:", PROJECT_ROOT)
print("Синтетика:", SYNTHETIC_CSV)
print("METR-LA:", METR_LA_H5)
print("MOEX:", MOEX_RAW_ROOT)
print("Результаты:", OUTPUT_ROOT)


Корень проекта: /home/loktar/Desktop/money_coursework
Синтетика: /home/loktar/Desktop/money_coursework/data/processed/synthetic_static.csv
METR-LA: /home/loktar/Desktop/money_coursework/data/raw/metr_la/metr-la.h5
MOEX: /home/loktar/Desktop/money_coursework/data/raw/moex_iss_2020_2025
Результаты: /home/loktar/Desktop/money_coursework/data/processed


## Общий экспорт

Для синтетики и METR-LA: первые 70% временных строк — train, следующие 10% — val, последние 20% — test.
Для MOEX: train до конца 2023 года, val — 2024, test — 2025. Перемешивания нет.

В CSV пропуски остаются пустыми. При загрузке `get_window()` заполняет **только вход**
прошлым наблюдением внутри выбранного окна истории; если его ещё нет — средним данной вершины,
оценённым только по train. Исходная маска возвращается отдельно. Цели не заполняются:
для обучения и метрик необходимо применять `y_mask`.

Первые val/test-примеры могут использовать законную историю из предыдущих частей.
Нормировка не выполнена: её параметры нужно обучить только на train.


## Чтение METR-LA

Сохраняем все датчики и исходный пятиминутный шаг. Неположительные значения
считаем ненаблюдаемыми по принятому для этого benchmark правилу; это не доказательство,
что каждый ноль является поломкой датчика. Географический граф пока не используется.


## Обработка MOEX

Функция `export_moex()` из `src/prepare_datasets.py` вызывает обычным импортом
обработчик `src/prepare_moex.py`: отбор обыкновенных акций TQBR только по train,
покрытие не менее 95%, затем 10 акций по медианному обороту, один выпуск на эмитента.

RV рассчитывается по полной сетке 52 десятиминутных свечей 10:00–18:40.
Сохраняются календарь, причины исключения дней и отчёт об отборе акций.
Временная папка используется только для промежуточных данных обработки.


## 1. Синтетика

Читаем указанный вами `data/processed/synthetic_static.csv`: `step` — время,
остальные столбцы — состояния вершин. Матрицу генератора по значениям не восстанавливаем.


In [2]:
if PROCESS_SYNTHETIC:
    synthetic_values, synthetic_segments, synthetic_audit = load_synthetic(
        SYNTHETIC_CSV, SYNTHETIC_TIME_COLUMN)
    results["synthetic"] = export_dataset(
        synthetic_values, synthetic_values, synthetic_segments,
        OUTPUT_ROOT / "synthetic", history=HISTORY,
        train_fraction=TRAIN_FRACTION, val_fraction=VAL_FRACTION,
        metadata=synthetic_audit,
    )
    print("Синтетика:", results["synthetic"]["rows"])
    print("Допустимые примеры:", results["synthetic"]["usable_samples"])


Синтетика: {'train': 4200, 'val': 600, 'test': 1201}
Допустимые примеры: {'train': 4140, 'val': 600, 'test': 1201}


## 2. METR-LA

Потребуется библиотека `tables`. Все исходные датчики сохраняются в исходном порядке.
Число датчиков не требуется приравнивать числу акций: сравнения моделей проводятся внутри каждого набора.


In [3]:
if PROCESS_METR_LA:
    metr_values, metr_audit = load_metr_la(METR_LA_H5)
    metr_segments = pd.Series(0, index=metr_values.index)
    results["metr_la"] = export_dataset(
        metr_values, metr_values, metr_segments,
        OUTPUT_ROOT / "metr_la", history=HISTORY,
        train_fraction=TRAIN_FRACTION, val_fraction=VAL_FRACTION,
        metadata=metr_audit,
    )
    print("METR-LA:", results["metr_la"]["rows"])
    print("Допустимые примеры:", results["metr_la"]["usable_samples"])


METR-LA: {'train': 23990, 'val': 3427, 'test': 6855}
Допустимые примеры: {'train': 22631, 'val': 3246, 'test': 6189}


## 3. Финансовые данные

Самая длительная обработка: проверка кэша, отбор акций на train, расчёт RV.
Сырая выгрузка сохраняется. Если подходящих акций меньше заданного числа, пороги автоматически не снижаются.


In [4]:
if PROCESS_MOEX:
    results["moex"] = export_moex(
        MOEX_RAW_ROOT, OUTPUT_ROOT / "moex", history=HISTORY,
        train_end=MOEX_TRAIN_END, val_end=MOEX_VAL_END, test_end=MOEX_TEST_END,
        n_stocks=MOEX_N_STOCKS, min_coverage=MOEX_MIN_COVERAGE,
        max_gap_days=MOEX_MAX_GAP_DAYS, epsilon=MOEX_LOG_EPSILON,
        run_id=MOEX_RUN_ID,
    )
    print("MOEX:", results["moex"]["node_order"])
    print("Строки:", results["moex"]["rows"])
    print("Допустимые примеры:", results["moex"]["usable_samples"])


Calendar: 250/2192 days
Calendar: 500/2192 days
Calendar: 750/2192 days
Calendar: 1000/2192 days
Calendar: 1250/2192 days
Calendar: 1500/2192 days
Calendar: 1750/2192 days
Calendar: 2000/2192 days
Calendar: 2192/2192 days
Train-only screening: 20/333
Train-only screening: 40/333
Train-only screening: 60/333
Train-only screening: 80/333
Train-only screening: 100/333
Train-only screening: 120/333
Train-only screening: 140/333
Train-only screening: 160/333
Train-only screening: 180/333
Train-only screening: 200/333
Train-only screening: 220/333
Train-only screening: 240/333
Train-only screening: 260/333
Train-only screening: 280/333
Train-only screening: 300/333
Train-only screening: 320/333
Train-only screening: 333/333
Aggregating selected security: SBER
Aggregating selected security: GAZP
Aggregating selected security: LKOH
Aggregating selected security: GMKN
Aggregating selected security: YNDX
Aggregating selected security: ROSN
Aggregating selected security: PLZL
Aggregating selected

## Как записаны CSV

Например, `data/processed/metr_la/train/series.csv`:

| time | датчик 1 | датчик 2 | … |
|---|---:|---:|---:|
| первый момент | значение | значение | … |
| следующий момент | значение | значение | … |

В каждой папке `train`, `val`, `test`:

- `series.csv`: один признак на вершину; исходные пропуски сохранены пустыми.
- `raw_values.csv`: значения в исходной шкале. Для MOEX это RV, тогда как в `series.csv` — log(RV).
- `observed_mask.csv`: 1 — наблюдение имеется, 0 — отсутствует.

В корне каждого набора: `schema.json` с порядком вершин, `calendar.csv`, `samples.csv`
с датами входов/целей и причинами исключения, `manifest.json` с настройками и контрольными суммами.
У MOEX также есть `reports/` с отбором акций и качеством данных.


In [5]:
summary_rows = []
for dataset, result in results.items():
    schema = json.loads((OUTPUT_ROOT / dataset / "schema.json").read_text())
    for split in SPLITS:
        summary_rows.append({"dataset": dataset, "split": split,
                             "rows": result["rows"][split],
                             "samples": result["usable_samples"][split],
                             "nodes": schema["n_nodes"], "features_per_node": 1})
    print(dataset, "готовность:", result["ready_for_training"])
    preview = pd.read_csv(OUTPUT_ROOT / dataset / "train/series.csv", nrows=3)
    display(preview.iloc[:, :6])
summary = pd.DataFrame(summary_rows)
display(summary)


synthetic готовность: True


,time,node_0,node_1,node_2,node_3,node_4
0,0,0.023914,-0.060859,-0.034834,0.007863,0.016628
1,1,0.132046,-0.050451,0.034699,0.050739,-0.067319
2,2,0.189874,-0.099603,-0.011444,0.022208,-0.061199


metr_la готовность: True


,time,773869,767541,767542,717447,717446
0,2012-03-01 00:00:00,64.375000,67.625000,67.125000,61.500000,66.875000
1,2012-03-01 00:05:00,62.666667,68.555556,65.444444,62.444444,64.444444
2,2012-03-01 00:10:00,64.000000,63.750000,60.000000,59.000000,66.500000


moex готовность: True


,time,SBER,GAZP,LKOH,GMKN,YNDX
0,2020-01-03 00:00:00,-9.028128,-9.149462,-9.781308,-9.235890,-9.219292
1,2020-01-06 00:00:00,-8.932237,-8.902652,-9.601863,-9.453578,-9.008901
2,2020-01-08 00:00:00,-8.905913,-9.798305,-8.556217,-8.987475,-9.082965


,dataset,split,rows,samples,nodes,features_per_node
0,synthetic,train,4200,4140,12,1
1,synthetic,val,600,600,12,1
2,synthetic,test,1201,1201,12,1
3,metr_la,train,23990,22631,207,1
4,metr_la,val,3427,3246,207,1
5,metr_la,test,6855,6189,207,1
6,moex,train,993,870,10,1
7,moex,val,256,252,10,1
8,moex,test,254,254,10,1


## Проверка одного примера для обучения

`get_window()` возвращает X формы `[60, N, 1]` и цель следующего момента формы `[N]`.
Это один пример; будущий DataLoader добавит размер батча.
Для METR-LA один шаг равен 5 минутам, для MOEX — следующей рыночной сессии, для синтетики — шагу генератора.

`X_mask` показывает исходные пропуски входа. `y_mask` обязательно используется при вычислении потерь и метрик.
`y_for_loss` содержит техническую заглушку на пропусках; она не становится наблюдаемой целью.
Здесь загружается только один пример каждого набора, а не огромный массив всех перекрывающихся окон.


In [6]:
examples = {}
for dataset, result in results.items():
    for split in SPLITS:
        if not result["usable_samples"][split]:
            print(dataset, split, "— нет допустимых примеров; проверьте samples.csv")
            continue
        example = get_window(OUTPUT_ROOT / dataset, split, 0)
        assert example["X"].shape == (HISTORY, len(result["node_order"]), 1)
        assert np.isfinite(example["X"]).all()
        assert example["y_mask"].any()
        assert example["origin_time"] < example["target_time"]
        examples[(dataset, split)] = example
        print(dataset, split, "X:", example["X"].shape, "y:", example["y"].shape,
              "наблюдаемых целей:", int(example["y_mask"].sum()),
              "из", len(example["y_mask"]))
        print("  история:", example["history_times"][0], "…", example["origin_time"],
              "→ цель:", example["target_time"])


synthetic train X: (60, 12, 1) y: (12,) наблюдаемых целей: 12 из 12
  история: 0 … 59 → цель: 60
synthetic val X: (60, 12, 1) y: (12,) наблюдаемых целей: 12 из 12
  история: 4140 … 4199 → цель: 4200
synthetic test X: (60, 12, 1) y: (12,) наблюдаемых целей: 12 из 12
  история: 4740 … 4799 → цель: 4800
metr_la train X: (60, 207, 1) y: (207,) наблюдаемых целей: 205 из 207
  история: 2012-03-01 00:00:00 … 2012-03-01 04:55:00 → цель: 2012-03-01 05:00:00
metr_la val X: (60, 207, 1) y: (207,) наблюдаемых целей: 205 из 207
  история: 2012-05-23 02:10:00 … 2012-05-23 07:05:00 → цель: 2012-05-23 07:10:00
metr_la test X: (60, 207, 1) y: (207,) наблюдаемых целей: 206 из 207
  история: 2012-06-03 23:45:00 … 2012-06-04 04:40:00 → цель: 2012-06-04 04:45:00
moex train X: (60, 10, 1) y: (10,) наблюдаемых целей: 10 из 10
  история: 2020-01-03 00:00:00 … 2020-03-31 00:00:00 → цель: 2020-04-01 00:00:00
moex val X: (60, 10, 1) y: (10,) наблюдаемых целей: 10 из 10
  история: 2023-10-09 00:00:00 … 2023-12-29